In [1]:
# my_data_collection.ipynb
# Сбор данных с камеры + управление моторами

# ============================================
# 1. ИМПОРТЫ И ЗАГРУЗКА
# ============================================
%run my_camera.ipynb
%run my_basic.ipynb

import os
import time
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime
import threading

# ============================================
# 2. НАСТРОЙКИ
# ============================================
OBJECT_NAME = "free"  # можно менять по мере сбора

# Корневая папка для датасета
DATASET_ROOT = "/home/jetsonbot/jetbot/notebooks/my_teleop/my_teleop_dataset"
SAVE_PATH = os.path.join(DATASET_ROOT, OBJECT_NAME)

# Создаем папку, если её нет
os.makedirs(SAVE_PATH, exist_ok=True)

print(f"📁 Кадры будут сохраняться в: {SAVE_PATH}")

# ============================================
# 3. ЗАПУСК КАМЕРЫ
# ============================================
camera = Camera()
camera.start()
time.sleep(1)

# ============================================
# 4. ВИДЖЕТ ДЛЯ ВИДЕО (СЛЕВА)
# ============================================
image_widget = widgets.Image(format='jpeg', width=480, height=360)

def update_video():
    while True:
        frame = camera.read()
        if frame is not None:
            frame_resized = cv2.resize(frame, (480, 360))
            frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
            _, jpeg = cv2.imencode('.jpg', frame_rgb, [cv2.IMWRITE_JPEG_QUALITY, 85])
            image_widget.value = jpeg.tobytes()
        time.sleep(0.03)

video_thread = threading.Thread(target=update_video)
video_thread.daemon = True
video_thread.start()

# ============================================
# 5. ПАНЕЛЬ УПРАВЛЕНИЯ СОХРАНЕНИЕМ (СПРАВА)
# ============================================
# Счетчик кадров
existing_files = [f for f in os.listdir(SAVE_PATH) if f.endswith('.jpg')]
next_number = len(existing_files) + 1

# Кнопка сохранения кадра
save_button = widgets.Button(
    description=f'📸 Сохранить [{next_number:03d}]',
    button_style='success',
    layout=widgets.Layout(width='200px', height='50px')
)

# Поле для ввода названия объекта
object_input = widgets.Text(
    value=OBJECT_NAME,
    description='Объект:',
    layout=widgets.Layout(width='200px')
)

# Кнопка смены объекта
change_object_button = widgets.Button(
    description='🔄 Сменить',
    button_style='info',
    layout=widgets.Layout(width='100px', height='30px')
)

# Информация о количестве
info_label = widgets.HTML(value=f"<b>Текущий объект:</b> {OBJECT_NAME}<br><b>Сохранено:</b> {len(existing_files)} кадров")

# Функция обновления путей
def update_object(new_name):
    global OBJECT_NAME, SAVE_PATH, next_number
    OBJECT_NAME = new_name
    SAVE_PATH = os.path.join(DATASET_ROOT, OBJECT_NAME)
    os.makedirs(SAVE_PATH, exist_ok=True)
    
    existing = [f for f in os.listdir(SAVE_PATH) if f.endswith('.jpg')]
    next_number = len(existing) + 1
    info_label.value = f"<b>Текущий объект:</b> {OBJECT_NAME}<br><b>Сохранено:</b> {len(existing)} кадров"
    save_button.description = f'📸 Сохранить [{next_number:03d}]'

# Обработчики
def on_save_clicked(b):
    global next_number
    frame = camera.read()
    if frame is not None:
        filename = f"{OBJECT_NAME}_{next_number:03d}.jpg"
        filepath = os.path.join(SAVE_PATH, filename)
        cv2.imwrite(filepath, frame)
        next_number += 1
        existing = [f for f in os.listdir(SAVE_PATH) if f.endswith('.jpg')]
        info_label.value = f"<b>Текущий объект:</b> {OBJECT_NAME}<br><b>Сохранено:</b> {len(existing)} кадров"
        save_button.description = f'📸 Сохранить [{next_number:03d}]'
        print(f"✅ Сохранен: {filename}")
    else:
        print("❌ Нет кадра")

def on_change_clicked(b):
    new_name = object_input.value.strip()
    if new_name:
        update_object(new_name)
        print(f"🔄 Объект изменен на: {new_name}")

save_button.on_click(on_save_clicked)
change_object_button.on_click(on_change_clicked)

# Сборка правой панели
right_panel = widgets.VBox([
    info_label,
    widgets.HBox([object_input, change_object_button]),
    save_button
], layout=widgets.Layout(margin='20px'))

# ============================================
# 6. ПАНЕЛЬ УПРАВЛЕНИЯ МОТОРАМИ (ВНИЗУ)
# ============================================
# Кнопки для моторов
motor_button_style = {'button_color': '#4CAF50'}

btn_left = widgets.Button(
    description='← ЛЕВО',
    button_style='',
    layout=widgets.Layout(width='100px', height='50px')
)

btn_right = widgets.Button(
    description='→ ПРАВО',
    button_style='',
    layout=widgets.Layout(width='100px', height='50px')
)

btn_forward = widgets.Button(
    description='↑ ВПЕРЁД',
    button_style='',
    layout=widgets.Layout(width='100px', height='50px')
)

btn_backward = widgets.Button(
    description='↓ НАЗАД',
    button_style='',
    layout=widgets.Layout(width='100px', height='50px')
)

btn_stop = widgets.Button(
    description='⏹ СТОП',
    button_style='danger',
    layout=widgets.Layout(width='80px', height='50px')
)


# Функции управления моторами
def forward_click(b):
    left_motor.value = 1
    right_motor.value = 1
    print("→ Вперёд")

def backward_click(b):
    left_motor.value = -1
    right_motor.value = -1
    print("← Назад")

def left_click(b):
    left_motor.value = -1
    right_motor.value = 1
    print("↺ Влево")

def right_click(b):
    left_motor.value = 1
    right_motor.value = -1
    print("↻ Вправо")

def stop_click(b):
    left_motor.value = 0
    right_motor.value = 0
    print("⏹ Стоп")

btn_forward.on_click(forward_click)
btn_backward.on_click(backward_click)
btn_left.on_click(left_click)
btn_right.on_click(right_click)
btn_stop.on_click(stop_click)

# Сборка панели моторов
motor_panel = widgets.HBox([
    btn_left, btn_right, btn_forward, btn_backward, btn_stop 
], layout=widgets.Layout(justify_content='center', margin='20px 0'))

# ============================================
# 7. ГЛАВНЫЙ ЭКРАН
# ============================================
# Верхняя часть: видео слева, панель сохранения справа
top_panel = widgets.HBox([
    image_widget,
    right_panel
], layout=widgets.Layout(justify_content='center', align_items='center'))

# Весь интерфейс
main_ui = widgets.VBox([
    top_panel,
    motor_panel
])

display(main_ui)

print("\n🎮 Управление:")
print("   - Кнопки моторов: движение по полу")
print("   - Правая панель: сбор данных")
print("   📸 Сохраняй кадры в 'free' когда можно ехать, и в 'blocked' когда есть препятствие")

📁 Кадры будут сохраняться в: /home/jetsonbot/jetbot/notebooks/my_teleop/my_teleop_dataset/free
Попытка открыть камеру с конвейером:
nvarguscamerasrc ! video/x-raw(memory:NVMM),width=640,height=480,framerate=30/1 ! nvvidconv ! video/x-raw,format=BGRx ! videoconvert ! video/x-raw,format=BGR ! appsink drop=1
✅ Камера успешно открыта через GStreamer.
📷 Поток захвата кадров запущен



🎮 Управление:
   - Кнопки моторов: движение по полу
   - Правая панель: сбор данных
   📸 Сохраняй кадры в 'free' когда можно ехать, и в 'blocked' когда есть препятствие


In [2]:
camera.stop()

🛑 Камера остановлена


In [3]:
import subprocess
import time

print("🔄 Сбрасываю демон камеры...")

# Для Python 3.6 используем stdout=PIPE вместо capture_output
result = subprocess.run(
    ["sudo", "systemctl", "restart", "nvargus-daemon"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

if result.returncode == 0:
    print("✅ Демон успешно перезапущен")
else:
    print(f"❌ Ошибка: {result.stderr.decode('utf-8')}")

time.sleep(2)
print("✅ Готово. Теперь перезапусти ячейку с камерой")

🔄 Сбрасываю демон камеры...
✅ Демон успешно перезапущен
✅ Готово. Теперь перезапусти ячейку с камерой
